# Predict Model – Using the Trained Models

This notebook loads the saved models from `train_model.ipynb` and shows how to predict the performance rating for employees.

- **Model A (all factors)** – reliable; use it for employees who have already joined and settled in
- **Model B (hiring time factors)** – saved for completeness, but it was **not better than guessing** in testing, so it must **not** be used for hiring decisions

Both models contain their own preprocessing, so employee data is passed in the same format as the cleaned data (text categories and numeric scores) – no manual encoding is needed.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
RAW_FILE = ROOT / "data" / "raw" / "INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "src" / "models"
FIGURES = ROOT / "src" / "visualization" / "figures"
TARGET = "PerformanceRating"
RANDOM_STATE = 42

import json
import joblib
from sklearn.model_selection import train_test_split

model_a = joblib.load(MODELS / "model_a_all_factors.joblib")
model_b = joblib.load(MODELS / "model_b_hiring_factors.joblib")
with open(MODELS / "model_metadata.json") as f:
    metadata = json.load(f)

FEATURES_A = metadata["model_a"]["features"]
FEATURES_B = metadata["model_b"]["features"]
LABELS = {int(k): v for k, v in metadata["class_labels"].items()}
data = pd.read_csv(PROCESSED / "employee_cleaned.csv")
print("Model A:", metadata["model_a"]["algorithm"], "| Model B:", metadata["model_b"]["algorithm"])

Model A: Gradient Boosting | Model B: Random Forest


## 1. A reusable prediction function

In [2]:
def predict_performance(employees, model="A"):
    """Predict the performance rating for one or more employees.

    employees : DataFrame with the factor columns used by the chosen model
    model     : "A" (all factors, recommended) or "B" (hiring time factors, not recommended)
    """
    pipeline, features = (model_a, FEATURES_A) if model == "A" else (model_b, FEATURES_B)
    missing = [c for c in features if c not in employees.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    inputs = employees[features]
    result = pd.DataFrame(index=employees.index)
    result["predicted_rating"] = pipeline.predict(inputs)
    result["predicted_label"] = result["predicted_rating"].map(LABELS)
    probabilities = pipeline.predict_proba(inputs)
    for position, rating in enumerate(pipeline.classes_):
        result[f"probability_rating_{rating}"] = probabilities[:, position].round(3)
    return result

## 2. Check on employees the model has never seen (the test set)

The same split as in `train_model.ipynb` is recreated (same random state), so these employees were **not** used for training or tuning.

In [3]:
train_index, test_index = train_test_split(data.index, test_size=0.2, stratify=data[TARGET], random_state=RANDOM_STATE)
test = data.loc[test_index]
predictions = predict_performance(test, model="A")
comparison = test[["EmpNumber", "EmpDepartment", TARGET]].join(predictions[["predicted_rating", "probability_rating_2", "probability_rating_3", "probability_rating_4"]])
comparison["correct"] = comparison[TARGET] == comparison["predicted_rating"]
print(f"Model A accuracy on {len(test)} unseen employees: {comparison['correct'].mean():.3f}")
comparison.head(10)

Model A accuracy on 240 unseen employees: 0.929


,EmpNumber,EmpDepartment,PerformanceRating,predicted_rating,probability_rating_2,probability_rating_3,probability_rating_4,correct
811,E100353,Development,3,3,0.005,0.992,0.004,True
1149,E100900,Development,3,3,0.008,0.990,0.002,True
662,E1002186,Sales,2,3,0.037,0.960,0.003,False
542,E1001967,Finance,2,2,0.862,0.131,0.007,True
858,E100429,Development,3,3,0.000,0.988,0.012,True
191,E1001338,Research & Development,3,3,0.003,0.989,0.008,True
479,E1001864,Research & Development,2,2,0.761,0.233,0.006,True
1,E1001006,Sales,3,3,0.004,0.993,0.003,True
659,E1002181,Research & Development,3,3,0.053,0.935,0.013,True
1136,E100877,Sales,3,3,0.006,0.991,0.003,True


In [4]:
pd.crosstab(comparison[TARGET], comparison["predicted_rating"], rownames=["actual rating"], colnames=["predicted rating"])

predicted rating,2,3,4
actual rating,,,
2,33,5,1
3,4,170,1
4,0,6,20


## 3. Predict for new example employees (Model A)

Three example employees are created from one existing employee record, changing only the factors that the analysis found most important. This shows how the prediction reacts to those factors.

In [5]:
template = data[FEATURES_A].iloc[0].to_dict()

def make_employee(**changes):
    row = dict(template)
    row.update(changes)
    return row

examples = pd.DataFrame([
    make_employee(EmpEnvironmentSatisfaction=1, EmpLastSalaryHikePercent=11, YearsSinceLastPromotion=5),
    make_employee(EmpEnvironmentSatisfaction=3, EmpLastSalaryHikePercent=14, YearsSinceLastPromotion=1),
    make_employee(EmpEnvironmentSatisfaction=4, EmpLastSalaryHikePercent=22, YearsSinceLastPromotion=0),
], index=["Example 1: low satisfaction, small hike, long wait for promotion",
          "Example 2: high satisfaction, average hike, recent promotion",
          "Example 3: very high satisfaction, high hike, just promoted"])
key_columns = ["EmpEnvironmentSatisfaction", "EmpLastSalaryHikePercent", "YearsSinceLastPromotion"]
examples[key_columns].join(predict_performance(examples, model="A"))

,EmpEnvironmentSatisfaction,EmpLastSalaryHikePercent,YearsSinceLastPromotion,predicted_rating,predicted_label,probability_rating_2,probability_rating_3,probability_rating_4
"Example 1: low satisfaction, small hike, long wait for promotion",1,11,5,2,Good,0.964,0.035,0.001
"Example 2: high satisfaction, average hike, recent promotion",3,14,1,3,Excellent,0.005,0.991,0.004
"Example 3: very high satisfaction, high hike, just promoted",4,22,0,4,Outstanding,0.001,0.020,0.979


## 4. Model B warning

Model B only uses information known at hiring. In testing its accuracy (about 64 percent) was **below the 73 percent** obtained by always predicting the most common rating, and it never identified an outstanding employee. It is shown here only so the difference in behaviour is visible.

In [6]:
b_test = predict_performance(test, model="B")
print("Model B accuracy on the same unseen employees:", round((b_test["predicted_rating"] == test[TARGET]).mean(), 3))
print("Accuracy of always predicting rating 3:", round((test[TARGET] == 3).mean(), 3))

Model B accuracy on the same unseen employees: 0.642
Accuracy of always predicting rating 3: 0.729


## 5. Scoring a file of employees

To score a spreadsheet or CSV of current employees, load it with the same column names as the cleaned data and call the function:

```python
new_employees = pd.read_excel("new_employees.xlsx")     # or pd.read_csv(...)
results = predict_performance(new_employees, model="A")
new_employees.join(results)
```

The columns needed are listed in `FEATURES_A` above (the same 26 factor columns as in the original data, without `EmpNumber` and `PerformanceRating`).